Deep Learning Models Training(CNN,CRNN)

This Notebook is used to train deep learning models to complete Music Genre Classification.
The training model includes:

-CNN (Convolutional Neural Network)
-CRNN (Convolutional Recurrent Neural Network)

The training data uses Mel Spectrogram obtained by preprocessing in advance, and evaluates the model performance in combination with 5-fold Cross Validation.
Finally, use the entire training set to retrain the best model and save the trained parameters for subsequent testing and model evaluation.

1 Import Libaraies
The following libraries are imported fordata processing,deep learning,cross validation and model evaluation.

In [ ]:
import os
import numpy as np

import torch
import torch.nn as nn
import pandas as pd

from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader, Subset
from model import CNN, CRNN

from sklearn.model_selection import KFold

2 Prepare Training Dataset
This part is intended to prepare the data which will be used for training the models.
Workflow:X_train.csv-->Read filenames-->y_train.csv-->Read corresponding genre labels-->Locate Mel Spectrogram (.npy)
      -->Load Mel Spectrogram-->Standardize-->Convert to Tensor
        

In [ ]:
#X_train.csv-->Read filenames-->y_train.csv-->Read corresponding genre labels
filename = x_df.iloc[i]["filename"]
genre = y_df.iloc[i]["genre"]
#Locate Mel Spectrogram (.npy)
file_path = os.path.join(mel_dir, genre, filename + ".npy")

#Load Mel Spectrogram
mel = np.load(file_path)
# Standardize each spectrogram
mel = (mel - mel.mean()) / (mel.std() + 1e-8)
#Convert to Tensor
mel = torch.tensor(mel, dtype=torch.float32)
mel = mel.unsqueeze(0)
label = torch.tensor(label, dtype = torch.long)

3 Training Configuration

Batch Size：The number of samples used for each update of parameters
    Batch Size = 32

Learning Rate：Adam Optimizer learning rate
    Learning Rate = 0.0005

Epoch：The number of training sets that the model has been completed
    Epoch = 50

Number of Classes：Genres in GTZAN dataset：10 Genres。
    Number of Classes = 10

Device：Autometically detect GPU
    if GPU is available：CUDA
    otherwise：CPU

In [ ]:
BATCH_SIZE = 32
LEARNING_RATE = 0.0005
EPOCHS = 50
NUM_CLASSES = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In order to improve the stability and classification performance of model training, this project adopts Adam Optimizer, CrossEntropy Loss and 5-Fold Cross Validation. The following is an introduction to their functions and reasons for selection.

Adam Optimizer：
Adam (Adaptive Moment Estimation) is a commonly used deep learning optimization algorithm, which combines Momentum (momentum) and Adaptive Learning Rate (Adaptive Learning Rate) Point.
Compared with the traditional random gradient descent (SGD), Adam can automatically adjust the learning rate according to different parameters, making the model training more stable and usually has a faster convergence speed. Therefore, this project chooses Adam Optimizer as the parameter optimization algorithm of CNN and CRNN.

In [ ]:
optimizer = Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

CrossEntropy Loss：
由于 GTZAN 数据集包含 10 个音乐流派（Genres），属于多分类问题（Multi-class Classification），因此采用 CrossEntropy Loss（交叉熵损失） 作为损失函数。
CrossEntropy Loss 能够衡量模型预测类别概率与真实标签之间的差异，当预测结果越接近真实类别时，损失值越小。

In [ ]:
criterion = nn.CrossEntropyLoss()

4 Load Traing Data
Load the pre-generated training feature and label files, then construct the PyTorch dataset for model training.

In [ ]:
import pandas as pd
Train_feature = "X_train.csv"
Train_label = "y_train.csv"
x_train = pd.read_csv(Train_feature)
y_train = pd.read_csv(Train_label)

train_dataset = GTZANDdataset(Mel_dir, x_train, y_train)

print("Dataset Information")
print(f"Training samples : {len(train_dataset)}")
labels = [label for _, label in train_dataset.samples]
print(f"Number of classes: {len(set(labels))}")
sample, label = train_dataset[0]
print(f"Input shape      : {sample.shape}")
print(f"Sample label     : {label}")

6 Cross Validation Setup
In order to improve the stability and classification performance of model training,our project adopts：5-Fold Cross Validation
Workflow：Training Dataset-->Split into 5 Folds-->4 Folds Training-->1 Fold Validation-->Repeat 5 Times
The macro F1 score is used as an evaluation indicator because it gives equal attention to each music genre.

In [ ]:
kf = StratifiedKFold(
    n_splits = 5, 
    shuffle = True, 
    random_state = 42
)

7 CNN and CRNN Training
Both CNN and CRNN adopt the same training strategies, including Adam optimizer, cross-entropy loss, five-fold hierarchical cross-verification, and final training for a complete training data set. The main difference is their network architecture. CNN focuses on spatial feature extraction, while CRNN additionally simulates time dependence.

7.1 Training Pipeline
This part introduces the training process of CNN and CRNN. CNN and CRNN use the same training process. ( In order to better compare the differences between CNN and CRNN in dealing with music genres)
The training data adopts the preprocessed Mel Spectrogram, and the 5-Fold Cross Validation is used to evaluate the model performance. During the training process, the model learns the characteristic representation of different music genres by constantly updating the parameters, so as to improve the classification accuracy.

In [ ]:
for images, labels in train_loader:
    
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs, dim=1)


-Adam Optimizer is adopted to update network parameters because it combines momentum and adaptive learning rates, leading to faster and more stable convergence.

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

-Cross Entropy Loss is used for multi-class classification by measuring the difference between predicted probabilities and the true labels.
Five-fold Cross Validation evaluates the model's generalization performance. The average Macro F1 Score across all folds is used as the model evaluation criterion.

In [ ]:
criterion = nn.CrossEntropyLoss()

-Five-fold Cross Validation evaluates the model's generalization performance. The average Macro F1 Score across all folds is used as the model evaluation criterion.

-Final Training
Use a complete training data set to retrain the selected model so that all available training samples contribute to the final model before testing.

In [ ]:
final_model = final_train(train_dataset, Model)

7.2 CNN and CRNN training
The Deep Learning models are trained using:
-Loss Function: CrossEntropyLoss
-Optimizer: Adam
-Learning Rate: 0.0005
-Batch Size: 32
-Number of Epochs: 50
-Evaluation metric: Macro F1-score and Accuracy
(Both CNN and CRNN use identical training settings to ensure a fair comparison. The only difference between the two models is their network architecture.)